# 03 – UDA-Hub agentic app
Runs the LangGraph multi-agent workflow (`agentic/workflow.py`). Without `OPENAI_API_KEY` (or with `UDAHUB_OFFLINE=1`) every agent uses its rule-based path; with a key the Classifier/Resolver/Escalation use the LLM.

Run notebooks 01 and 02 first.

In [1]:
import os, uuid
from agentic import config
from agentic.workflow import create_app, run_ticket, inspect_thread
from agentic.logging_utils import search_logs
from utils import chat_interface, show
print('offline (rule-based) mode:', config.offline_mode())
app = create_app()
print(app.get_graph().draw_mermaid())

offline (rule-based) mode: True
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	intake(intake)
	load_memory(load_memory)
	classifier(classifier)
	supervisor(supervisor)
	resolver(resolver)
	account_ops(account_ops)
	escalation(escalation)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> intake;
	account_ops -.-> escalation;
	account_ops -.-> finalize;
	classifier --> supervisor;
	escalation --> finalize;
	intake --> load_memory;
	load_memory --> classifier;
	resolver -.-> escalation;
	resolver -.-> finalize;
	supervisor -.-> account_ops;
	supervisor -.-> escalation;
	supervisor -.-> resolver;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 1. Successful resolution from the knowledge base

In [2]:
show(run_ticket(app, 'How do I reset my password?'), 'How do I reset my password?')

CUSTOMER: How do I reset my password?
  route=resolver (question about account -> knowledge-based resolution) status=resolved confidence=0.3729
  AGENT: Hi there,

According to our help article "Reset your password or fix login problems": On the login screen tap Forgot Password and enter your email. We send a reset link that is valid for 30 minutes; check your spam folder if it does not arrive. After 5 failed attempts your account is locked for 15 minutes.

Does that answer your question?



## 2. Escalation: no relevant article / low confidence

In [3]:
for q in ['Do you sell gift cards for my cat?', "What's the weather like in Paris?"]:
    show(run_ticket(app, q), q)

CUSTOMER: Do you sell gift cards for my cat?
  route=resolver (question about billing -> knowledge-based resolution) status=escalate confidence=0.1877
  AGENT: I've passed this to our billing team as P3 (reference tkt-033bc3bca5). You'll hear back within 1 business day via this conversation.



CUSTOMER: What's the weather like in Paris?
  route=resolver (question about other -> knowledge-based resolution) status=escalate confidence=0.0
  AGENT: I've passed this to our general support team as P3 (reference tkt-406e6971b5). You'll hear back within 1 business day via this conversation.



## 3. Routing from ticket content + metadata (security, human request, stale urgent ticket)

In [4]:
from datetime import datetime, timedelta
show(run_ticket(app, 'Someone hacked my account and made unauthorized charges!'), 'hacked...')
show(run_ticket(app, 'I want to talk to a real person'), 'human please')
old = (datetime.utcnow() - timedelta(hours=72)).isoformat()
show(run_ticket(app, 'I need help with my invoice today, it is urgent', created_at=old, channel='email'), 'urgent, ticket 72h old')

CUSTOMER: hacked...
  route=escalation (security risk (possible fraud/unauthorised access)) status=escalate confidence=None
  AGENT: I've passed this to our trust and safety team as P1 (reference tkt-93c7febfab). You'll hear back within 1 hour via this conversation.

CUSTOMER: human please
  route=escalation (customer asked for a human) status=escalate confidence=None
  AGENT: I've passed this to our general support team as P3 (reference tkt-23118f3045). You'll hear back within 1 business day via this conversation.

CUSTOMER: urgent, ticket 72h old
  route=escalation (high-urgency ticket open for more than 48h) status=escalate confidence=None
  AGENT: I've passed this to our billing team as P2 (reference tkt-bdb01cfc56). You'll hear back within 4 business hours via this conversation.



/tmp/ipykernel_508/2139918280.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  old = (datetime.utcnow() - timedelta(hours=72)).isoformat()


## 4. Tool use, multi-step state and short-term memory (one `thread_id`)

In [5]:
tid = None
for msg in ['Please cancel my subscription', 'bob@example.com', 'yes']:
    out = run_ticket(app, msg, tid); tid = out['thread_id']; show(out, msg)
import pprint
info = inspect_thread(app, tid)
pprint.pprint([(u['tool'], u['ok']) for u in info['tool_usage']])
pprint.pprint(info['messages'])

CUSTOMER: Please cancel my subscription
  route=account_ops (customer wants an account action (cancel_subscription)) status=awaiting_customer confidence=1.0
  AGENT: I can help with that. To find your account, could you please share the email address you use for CultPass?



CUSTOMER: bob@example.com
  route=account_ops (continuing a multi-step account operation) status=awaiting_customer confidence=1.0
  AGENT: Bob, you're on the Basic plan. Do you want me to cancel it? Reply YES to confirm.



CUSTOMER: yes
  route=account_ops (continuing a multi-step account operation) status=resolved confidence=1.0
  AGENT: Done, Bob. Your subscription has been cancelled and you won't be charged again. Reservations you already made stay valid.

[('lookup_account', True),
 ('get_subscription_status', True),
 ('cancel_subscription', True)]
[('human', 'Please cancel my subscription'),
 ('ai',
  'I can help with that. To find your account, could you please share the '
  'email address you use for CultPass?'),
 ('human', 'bob@example.com'),
 ('ai',
  "Bob, you're on the Basic plan. Do you want me to cancel it? Reply YES to "
  'confirm.'),
 ('human', 'yes'),
 ('ai',
  "Done, Bob. Your subscription has been cancelled and you won't be charged "
  'again. Reservations you already made stay valid.')]


## 5. Refund: auto-approved, denied by policy, needs approval (escalated)

In [6]:
for user in ['u_grace', 'u_bob', 'u_dev']:
    out = run_ticket(app, 'I want a refund', external_user_id=user)
    if out['resolution']['status'] == 'awaiting_customer':
        out = run_ticket(app, 'yes', out['thread_id'])
    print('###', user); show(out, 'I want a refund')

### u_grace
CUSTOMER: I want a refund
  route=account_ops (continuing a multi-step account operation) status=resolved confidence=1.0
  AGENT: Done, Grace. I've refunded $29.99 (ref rf_5b103d49). It reaches your card in 5 to 10 business days.



### u_bob
CUSTOMER: I want a refund
  route=account_ops (customer wants an account action (request_refund)) status=resolved confidence=1.0
  AGENT: I'm sorry, Bob: The payment is older than the 14-day refund window. You can see the full policy in our Refund policy article.



### u_dev
CUSTOMER: I want a refund
  route=account_ops (customer wants an account action (request_refund)) status=escalate confidence=1.0
  AGENT: Dev, your payment of $299.99 is above the amount I can refund automatically. I've logged the request (rf_3ce436eb) and a billing specialist will review it.

I've passed this to our billing team as P2 (reference tkt-abbd95ce8d). You'll hear back within 4 business hours via this conversation.



## 6. Long-term memory across sessions (new thread, same customer)

In [7]:
first = run_ticket(app, 'How do I update my payment method? Please keep it short.', external_user_id='u_alice'); show(first)
second = run_ticket(app, 'My card changed again, how do I update the payment method?', external_user_id='u_alice')
assert second['thread_id'] != first['thread_id']; show(second)
print(second['memory_context']['preferences'])

  route=resolver (question about billing -> knowledge-based resolution) status=resolved confidence=0.4309
  AGENT: Hi Alice,

According to our help article "How to update your payment method": Open Settings > Billing > Payment Method and tap Add New Card. Enter the card details and set it as default. Your next renewal is charged to the default card.

Does that answer your question?



  route=resolver (question about billing -> knowledge-based resolution) status=resolved confidence=0.4708
  AGENT: Hi Alice,

According to our help article "How to update your payment method": Open Settings > Billing > Payment Method and tap Add New Card. Your next renewal is charged to the default card.

(I can see you contacted us about something similar on 2026-10-17. If those steps didn't fix it, tell me and I'll bring in a specialist.)

Does that answer your question?

{'response_style': 'concise'}


## 7. Structured, searchable logs

In [8]:
tid2 = second['ticket_id']
for r in search_logs(ticket_id=tid2): print(r['event'], {k: v for k, v in r.items() if k not in ('ts','event','ticket_id','thread_id')})

agent_decision {'agent': 'intake', 'decision': 'ticket_received', 'channel': 'chat', 'user_known': True, 'text_preview': 'My card changed again, how do I update the payment method?'}
agent_decision {'agent': 'memory', 'decision': 'memory_loaded', 'known_customer': True, 'preferences': ['response_style'], 'similar_past_issues': 1, 'prior_tickets': 1, 'open_escalations': 0}
agent_decision {'agent': 'classifier', 'decision': 'classified', 'source': 'rules', 'category': 'billing', 'intent': 'question', 'action': 'none', 'urgency': 'medium', 'sentiment': 'negative', 'complexity': 'simple', 'flags': []}
agent_decision {'agent': 'supervisor', 'decision': 'route', 'route': 'resolver', 'reason': 'question about billing -> knowledge-based resolution', 'complexity': 'simple'}
tool_call {'agent': 'resolver', 'tool': 'search_knowledge_base', 'args': {'account_id': 'cultpass', 'query': 'My card changed again, how do I update the payment method?'}, 'ok': True, 'error_code': None}
agent_decision {'age

## 8. Interactive chat (run locally; `/quit` to exit)

In [9]:
# chat_interface(app, external_user_id='u_alice')   # or external_user_id=None for an anonymous customer